# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set IDs and show their fields
print("Available Record Sets and their Fields (using @id):\n")
record_sets = list(dataset.metadata.record_sets)
record_set_ids = []
record_set_columns = {}
for rs in record_sets:
    print(f"Record Set: {rs['@id']} - name: {rs.get('name', None)}")
    record_set_ids.append(rs['@id'])
    print("  Fields (by @id):")
    col_ids = []
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - {field['@id']} (name: {field.get('name', None)})")
            col_ids.append(field['@id'])
    record_set_columns[rs['@id']] = col_ids
    print()
if not record_sets:
    print("[No record sets found directly in Croissant metadata. Some Croissant schemas may define fields at other levels or via file-level access.]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If the dataset has no literal record_set in metadata, try loading default/main record set
if record_set_ids:
    to_extract = record_set_ids
else:
    # Try loading a default (first) source
    to_extract = []
    print("No explicit record sets found in metadata. Attempting to load default records...")

dataframes = {}
for record_set in to_extract:
    print(f"Loading data for record set {record_set}")
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# For demonstration, show the columns of the first available DataFrame/record set
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for {sample_record_set_id}:\n{dataframes[sample_record_set_id].columns.tolist()}")
    display(dataframes[sample_record_set_id].head())
else:
    print("No DataFrames created, likely due to missing record set definitions in the Croissant metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
if dataframes:
    df = dataframes[sample_record_set_id]
    # Attempt to pick a numeric field by inspecting dtypes or column names
    import numpy as np
    # Try to find typical numeric field, fallback to first float/int column
    numeric_field = None
    for c in df.columns:
        # common MS column names
        if 'abundance' in c.lower() or 'count' in c.lower() or 'coverage' in c.lower() or 'mw' in c.lower():
            if np.issubdtype(df[c].dtype, np.number):
                numeric_field = c
                break
    if numeric_field is None:
        nums = df.select_dtypes(include=np.number).columns.tolist()
        if nums:
            numeric_field = nums[0]

    if numeric_field is not None:
        print(f"Using numeric field for EDA: '{numeric_field}'")
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a field that looks categorical
        group_field = None
        for c in df.columns:
            if 'sample' in c.lower() or 'group' in c.lower() or 'category' in c.lower():
                if not np.issubdtype(df[c].dtype, np.number):
                    group_field = c
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("Dataset not loaded or no DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dataframes and numeric_field is not None:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If we did a categorical group, boxplot
    if group_field:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("Visualization cannot be produced; dataset or numeric_field unavailable.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded from the Croissant schema and record sets inspected.
- Basic exploratory data analysis and visualization were applied to fields such as protein abundance or counts (where available).
- For further work, refine field selection based on domain-specific questions, or consult the dataset's [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for more details.